# Forecast performance

Reads a run of `uv run forecast run` from `runs/forecast_*/`: the report (`ForecastReport`), the quantile forecasts, the copula scenarios and the backtests driven by each forecast.

In [ ]:
from datetime import timedelta

import numpy as np
import plotly.graph_objects as go
import polars as pl
from plotly.subplots import make_subplots

from battery_optimisation.columns import Col
from battery_optimisation.data.nems import NemsSeries, load_nems
from battery_optimisation.forecast.quantiles import pit
from battery_optimisation.forecast.report import list_forecast_runs, load_forecast_run

runs = list_forecast_runs()
[p.name for p in runs]

In [ ]:
run = load_forecast_run(runs[0])  # most recent; or pick by name
report, qf, scen = run.report, run.quantiles, run.scenarios
levels = report.config.quantile_levels
realised = load_nems(
    report.test_start - timedelta(days=1), report.test_end, series=[NemsSeries.USEP]
).select(Col.INTERVAL_START, Col.USEP)
o, cfg = report.overall, report.config
covs = [str(x) for x in cfg.covariates]
print(run.path.name)
print(f"{report.issues} daily issues, {report.test_start} -> {report.test_end}")
print(f"covariates {covs}, weather {cfg.weather.model}")
print(
    f"CRPS {o.crps:.1f} | MAE(median) {o.mae_median:.1f} | MASE {o.mase:.2f} "
    f"(scale: seasonal-naive MAE {report.mase_scale:.1f}) | coverage 10-90 {o.coverage_10_90:.1%}"
)
print(
    f"spikes >= {report.spike_threshold:.0f}: hit {o.spike_hit_rate:.1%}, "
    f"false alarm {o.spike_false_alarm:.1%}"
)
pl.DataFrame([c.model_dump() for c in report.capture]).select(
    "strategy",
    "horizon",
    pl.col("profit_sgd").round(0),
    pl.col("cycles_per_day").round(2),
    pl.col("capture_vs_perfect_full").round(3),
    pl.col("capture_vs_perfect_day").round(3),
)

## By month

CRPS and MAE in SGD/MWh (lower is better). MASE = MAE of the median / MAE of the seasonal-naive forecast (same interval yesterday) on the history before the test: below 1 beats naive. Coverage of the 10-90% band should sit near 80%.

In [ ]:
monthly = pl.DataFrame([m.model_dump() for m in report.monthly])
x = monthly["period_start"]
fig = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.06)
fig.add_bar(x=x, y=monthly["crps"], name="CRPS", row=1, col=1)
fig.add_bar(x=x, y=monthly["mae_median"], name="MAE (median)", row=1, col=1)
fig.add_scatter(x=x, y=monthly["mase"], name="MASE", row=2, col=1)
fig.add_hline(y=1.0, line_dash="dot", row=2, col=1)
fig.add_scatter(x=x, y=monthly["coverage_10_90"], name="coverage 10-90", row=3, col=1)
fig.add_hline(y=0.8, line_dash="dot", row=3, col=1)
fig.update_yaxes(title_text="SGD/MWh", row=1, col=1)
fig.update_yaxes(title_text="MASE", row=2, col=1)
fig.update_yaxes(title_text="share", tickformat=".0%", row=3, col=1)
fig.update_layout(
    height=700, barmode="group", hovermode="x unified", title="Forecast accuracy by month"
)

## One day: quantile fan, scenarios and realised USEP

Defaults to the day with the highest realised price; set `day` to look elsewhere.

In [ ]:
day = (
    realised.filter(pl.col(Col.INTERVAL_START).dt.date() >= report.test_start)
    .sort(Col.USEP)[-1, Col.INTERVAL_START]
    .date()
)
# day = date(2026, 3, 2)

q = (
    qf.filter(pl.col(Col.ISSUED_AT).dt.date() == day)
    .pivot(on=Col.QUANTILE, index=Col.INTERVAL_START, values=Col.USEP_FORECAST)
    .sort(Col.INTERVAL_START)
)
y = realised.filter(pl.col(Col.INTERVAL_START).dt.date() == day)
x = q[Col.INTERVAL_START]
fig = go.Figure()
for lo, hi, alpha in [(0.05, 0.95, 0.15), (0.1, 0.9, 0.25), (0.25, 0.75, 0.4)]:
    fig.add_scatter(
        x=x, y=q[str(hi)], mode="lines", line_width=0, showlegend=False, hoverinfo="skip"
    )
    fig.add_scatter(
        x=x,
        y=q[str(lo)],
        mode="lines",
        line_width=0,
        fill="tonexty",
        fillcolor=f"rgba(42,157,143,{alpha})",
        name=f"{lo:.0%}-{hi:.0%}",
    )
fig.add_scatter(x=x, y=q["0.5"], mode="lines", line_color="#2a9d8f", name="median")
fig.add_scatter(
    x=y[Col.INTERVAL_START],
    y=y[Col.USEP],
    mode="lines+markers",
    line_color="#264653",
    name="realised",
)
fig.update_layout(
    title=f"Forecast issued {day} 00:00 vs realised", yaxis_title="SGD/MWh", hovermode="x unified"
)

In [ ]:
s = scen.filter(pl.col(Col.ISSUED_AT).dt.date() == day)
fig = go.Figure()
w_max = s[Col.WEIGHT].max()
for (k,), path in s.group_by(Col.SCENARIO, maintain_order=True):
    w = path[0, Col.WEIGHT]
    fig.add_scatter(
        x=path[Col.INTERVAL_START],
        y=path[Col.USEP_FORECAST],
        mode="lines",
        showlegend=False,
        line={
            "color": f"rgba(231,111,81,{0.15 + 0.85 * w / w_max:.2f})",
            "width": 1 + 3 * w / w_max,
        },
        hovertemplate=f"scenario {k}, weight {w:.3f}<extra></extra>",
    )
fig.add_scatter(
    x=y[Col.INTERVAL_START], y=y[Col.USEP], mode="lines", line_color="#264653", name="realised"
)
fig.update_layout(
    title=f"{s[Col.SCENARIO].n_unique()} copula scenarios for {day} (width/opacity = weight)",
    yaxis_title="SGD/MWh",
)

## Calibration

PIT values of the realised prices should be uniform (flat histogram); the observed share below each quantile should match its level (points on the diagonal).

In [ ]:
test_qf = qf.filter(pl.col(Col.ISSUED_AT).dt.date() >= report.test_start).sort(
    Col.ISSUED_AT, Col.LEAD_PERIODS, Col.QUANTILE
)
values = test_qf[Col.USEP_FORECAST].to_numpy().reshape(-1, len(levels))
stamps = test_qf.filter(pl.col(Col.QUANTILE) == levels[0])[Col.INTERVAL_START]
yv = (
    pl.DataFrame({Col.INTERVAL_START: stamps})
    .join(realised, on=Col.INTERVAL_START, how="left")[Col.USEP]
    .to_numpy()
)
ok = ~np.isnan(yv)
u = pit(levels, values[ok], yv[ok])
observed = [(yv[ok] <= values[ok, i]).mean() for i in range(len(levels))]
fig = make_subplots(rows=1, cols=2, subplot_titles=["PIT histogram", "Reliability"])
fig.add_histogram(x=u, nbinsx=20, histnorm="probability density", row=1, col=1, showlegend=False)
fig.add_hline(y=1.0, line_dash="dot", row=1, col=1)
fig.add_scatter(x=levels, y=observed, mode="markers+lines", row=1, col=2, showlegend=False)
fig.add_scatter(x=[0, 1], y=[0, 1], mode="lines", line_dash="dot", row=1, col=2, showlegend=False)
fig.update_xaxes(title_text="nominal level", row=1, col=2)
fig.update_yaxes(title_text="observed share below", row=1, col=2)
fig.update_layout(height=400)

## Error by time of day

With a 00:00 issue, the lead time equals the time of day: later intervals are forecast further ahead.

In [ ]:
by_lead = (
    test_qf.filter(pl.col(Col.QUANTILE) == 0.5)
    .join(realised, on=Col.INTERVAL_START)
    .group_by(Col.LEAD_PERIODS)
    .agg(
        (pl.col(Col.USEP_FORECAST) - pl.col(Col.USEP)).abs().mean().alias("mae"),
        (pl.col(Col.USEP_FORECAST) - pl.col(Col.USEP)).mean().alias("bias"),
        pl.col(Col.USEP).mean().alias("mean_usep"),
    )
    .sort(Col.LEAD_PERIODS)
    .with_columns(((pl.col(Col.LEAD_PERIODS) - 1) / 2).alias("hour"))
)
fig = go.Figure(
    [
        go.Bar(x=by_lead["hour"], y=by_lead["mae"], name="MAE (median)"),
        go.Scatter(x=by_lead["hour"], y=by_lead["bias"], name="bias (forecast - realised)"),
        go.Scatter(x=by_lead["hour"], y=by_lead["mean_usep"], name="mean USEP", line_dash="dot"),
    ]
)
fig.update_layout(xaxis_title="hour of day (SGT)", yaxis_title="SGD/MWh", hovermode="x unified")

## Value of the forecast: profit by month and strategy

Each strategy plans every day at 00:00 with the MILP on its own forecast and self-schedules the plan; perfect foresight (full horizon) is the upper bound.

In [ ]:
rows = []
for (name, horizon), (summary, _) in run.backtests.items():
    for m in summary.monthly:
        rows.append(
            {"strategy": f"{name} ({horizon})", "month": m.period_start, "profit_sgd": m.profit_sgd}
        )
profit = pl.DataFrame(rows)
go.Figure(
    [
        go.Bar(x=g["month"], y=g["profit_sgd"], name=k)
        for (k,), g in profit.group_by("strategy", maintain_order=True)
    ]
).update_layout(barmode="group", yaxis_title="SGD", title="Monthly profit by strategy")

In [ ]:
# Copula check (CLAUDE.md): lag-1 correlation implied by the copula vs measured on the month itself
pl.DataFrame([s.model_dump() for s in report.scenarios]).with_columns(
    pl.col("copula_lag1", "realised_lag1").round(3)
)